In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import re
import nltk
nltk.download('punkt')
nltk.download('stopwords')

print("Required libraries and NLTK data imported successfully.")

Required libraries and NLTK data imported successfully.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
print("All required libraries and NLTK data have been successfully imported and are ready for use.")

All required libraries and NLTK data have been successfully imported and are ready for use.


The first step is to load the `data.csv` file into a pandas DataFrame from the specified Google Drive path and then display its head to inspect the raw data.



In [ ]:
df = pd.read_csv('/content/drive/MyDrive/data.csv')
print("DataFrame loaded successfully. Displaying the first few rows:")
df.head()

DataFrame loaded successfully. Displaying the first few rows:


,diseases,anxiety_and_nervousness,depression,shortness_of_breath,depressive_or_psychotic_symptoms,sharp_chest_pain,dizziness,insomnia,abnormal_involuntary_movements,chest_tightness,...,stuttering_or_stammering,problems_with_orgasm,nose_deformity,lump_over_jaw,sore_in_nose,hip_weakness,back_swelling,ankle_stiffness_or_tightness,ankle_weakness,neck_weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0



Since the dataset does not explicitly contain a 'symptom' column but has numerous binary columns representing symptoms, I will first create a 'symptom_text' column by concatenating the names of all present symptoms (where the value is 1) for each row. Then, I will define a text preprocessing function and apply it to this newly created 'symptom_text' column to generate the `cleaned_symptom` column, as specified in the instructions. Finally, I will display the relevant columns to verify the preprocessing.



In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Re-download NLTK data to ensure it's available for this cell's execution
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True) # Explicitly download 'punkt_tab' as indicated by the error

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^A-z ]', '', text) # Remove punctuation, keeping only letters and spaces
    text = re.sub(r'\d+', '', text)    # Remove numbers
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered_tokens)

# Identify symptom columns (all columns except 'diseases')
symptom_columns = df.columns.drop('diseases')

# Create a 'symptom_text' column by concatenating symptom names where value is 1
df['symptom_text'] = df.apply(lambda row: ' '.join([col for col in symptom_columns if row[col] == 1]), axis=1)

# Apply the preprocess_text function to the new 'symptom_text' column
df['cleaned_symptom'] = df['symptom_text'].apply(preprocess_text)

print("Original 'symptom_text' column and 'cleaned_symptom' column created. Displaying first few rows:")
df[['diseases', 'symptom_text', 'cleaned_symptom']].head()

Original 'symptom_text' column and 'cleaned_symptom' column created. Displaying first few rows:


,diseases,symptom_text,cleaned_symptom
0,panic disorder,anxiety_and_nervousness shortness_of_breath de...,anxiety_and_nervousness shortness_of_breath de...
1,panic disorder,shortness_of_breath depressive_or_psychotic_sy...,shortness_of_breath depressive_or_psychotic_sy...
2,panic disorder,anxiety_and_nervousness depression shortness_o...,anxiety_and_nervousness depression shortness_o...
3,panic disorder,anxiety_and_nervousness depressive_or_psychoti...,anxiety_and_nervousness depressive_or_psychoti...
4,panic disorder,anxiety_and_nervousness depression insomnia ab...,anxiety_and_nervousness depression insomnia ab...


In [ ]:
from sklearn.preprocessing import LabelEncoder
import collections

# 1. Convert 'diseases' column to numerical labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['diseases'])
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

print("Disease labels encoded successfully. Label mapping:")
print(label_mapping)

# 2. Create a vocabulary by tokenizing the 'cleaned_symptom' column
all_words = []
for text in df['cleaned_symptom']:
    all_words.extend(text.split())

word_counts = collections.Counter(all_words)
unique_words = sorted(word_counts.keys())

vocabulary = {"<pad>": 0, "<unk>": 1}
for i, word in enumerate(unique_words):
    vocabulary[word] = i + 2  # Start indexing from 2 after <pad> and <unk>

print(f"\nVocabulary created with {len(vocabulary)} unique words.")
print("First 10 items in vocabulary:", list(vocabulary.items())[:10])

Disease labels encoded successfully. Label mapping:
{'Rare Disease': np.int64(0), 'abdominal aortic aneurysm': np.int64(1), 'abdominal hernia': np.int64(2), 'abscess of nose': np.int64(3), 'abscess of the lung': np.int64(4), 'abscess of the pharynx': np.int64(5), 'acanthosis nigricans': np.int64(6), 'acariasis': np.int64(7), 'achalasia': np.int64(8), 'acne': np.int64(9), 'actinic keratosis': np.int64(10), 'acute bronchiolitis': np.int64(11), 'acute bronchitis': np.int64(12), 'acute bronchospasm': np.int64(13), 'acute fatty liver of pregnancy (aflp)': np.int64(14), 'acute glaucoma': np.int64(15), 'acute kidney injury': np.int64(16), 'acute otitis media': np.int64(17), 'acute pancreatitis': np.int64(18), 'acute respiratory distress syndrome (ards)': np.int64(19), 'acute sinusitis': np.int64(20), 'acute stress reaction': np.int64(21), 'adhesive capsulitis of the shoulder': np.int64(22), 'adjustment reaction': np.int64(23), 'adrenal adenoma': np.int64(24), 'alcohol abuse': np.int64(25), 'a

In [ ]:
import torch
from sklearn.model_selection import train_test_split

# 3. Convert 'cleaned_symptom' to numerical ID sequences and pad
def text_to_sequence(text, vocabulary, max_len):
    words = text.split()
    sequence = [vocabulary.get(word, vocabulary['<unk>']) for word in words]
    if len(sequence) < max_len:
        sequence = sequence + [vocabulary['<pad>']] * (max_len - len(sequence))
    else:
        sequence = sequence[:max_len]
    return sequence

# Using the max_sequence_length identified from previous analysis or setting a reasonable one
# Let's re-calculate max_sequence_length in case it was not explicitly saved or for robustness
symptom_lengths = [len(text.split()) for text in df['cleaned_symptom']]
max_sequence_length = max(symptom_lengths) if symptom_lengths else 0
print(f"Maximum sequence length found: {max_sequence_length}")

# If max_sequence_length is too large, we might want to cap it. Let's cap it at a reasonable value if it exceeds a certain threshold, e.g., 50
# For this dataset, the max length is likely small, but for generality:
# if max_sequence_length > 50:
#     max_sequence_length = 50

X = [text_to_sequence(text, vocabulary, max_sequence_length) for text in df['cleaned_symptom']]
y = df['label'].values

print(f"Converted {len(X)} symptom sequences to numerical IDs and padded to length {max_sequence_length}.")

# 4. Split the dataset into training, validation, and test sets
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val) # 0.25 of 0.8 is 0.2

print(f"Dataset split into training: {len(X_train)}, validation: {len(X_val)}, test: {len(X_test)} samples.")

# 5. Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print("Features and labels converted to PyTorch tensors.")

print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_val:", X_val.shape)
print("Shape of y_val:", y_val.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Maximum sequence length found: 12
Converted 246945 symptom sequences to numerical IDs and padded to length 12.
Dataset split into training: 148167, validation: 49389, test: 49389 samples.
Features and labels converted to PyTorch tensors.

Shape of X_train: torch.Size([148167, 12])
Shape of y_train: torch.Size([148167])
Shape of X_val: torch.Size([49389, 12])
Shape of y_val: torch.Size([49389])
Shape of X_test: torch.Size([49389, 12])
Shape of y_test: torch.Size([49389])


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# Define the neural network model
class SymptomClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, max_seq_len, dropout_prob=0.5):
        super(SymptomClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0) # padding_idx=0 for <pad>
        self.flatten = nn.Flatten()
        # The input dimension to the first linear layer will be max_seq_len * embedding_dim
        self.fc1 = nn.Linear(max_seq_len * embedding_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        # text = [batch size, max_seq_len]
        embedded = self.embedding(text)
        # embedded = [batch size, max_seq_len, embedding_dim]
        flattened = self.flatten(embedded)
        # flattened = [batch size, max_seq_len * embedding_dim]
        x = self.fc1(flattened)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Model parameters
VOCAB_SIZE = len(vocabulary)
EMBEDDING_DIM = 128
HIDDEN_DIM = 256 # Example hidden dimension
OUTPUT_DIM = len(label_mapping) # Number of unique disease labels
MAX_SEQ_LEN = max_sequence_length # From previous step

# Instantiate the model
model = SymptomClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, MAX_SEQ_LEN)

# Print the model architecture and parameter count
print("Model Architecture:")
print(model)

# Count trainable parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\nThe model has {count_parameters(model):,} trainable parameters.')


Model Architecture:
SymptomClassifier(
  (embedding): Embedding(330, 128, padding_idx=0)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=1536, out_features=256, bias=True)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=256, out_features=713, bias=True)
)

The model has 619,465 trainable parameters.


In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Move model to device
model.to(device)

# 2. Define hyperparameters
LEARNING_RATE = 0.001
BATCH_SIZE = 64
N_EPOCHS = 30

# 3. Initialize the Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 4. Define the loss function
criterion = nn.CrossEntropyLoss()

# 5. Create TensorDatasets and DataLoaders
train_data = TensorDataset(X_train, y_train)
val_data = TensorDataset(X_val, y_val)
test_data = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_data, shuffle=True, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_data, shuffle=False, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, shuffle=False, batch_size=BATCH_SIZE)

print(f"DataLoaders created with batch size {BATCH_SIZE}.")

# 6. Implement the training loop
def train(model, iterator, optimizer, criterion, device):
    epoch_loss = 0
    epoch_acc = 0
    model.train() # Set the model to training mode

    for X_batch, y_batch in iterator:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad() # Zero the gradients
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        # Calculate accuracy
        _, predicted_labels = torch.max(predictions, 1)
        correct_predictions = (predicted_labels == y_batch).sum().item()
        accuracy = correct_predictions / y_batch.shape[0]

        loss.backward() # Backpropagate the loss
        optimizer.step() # Update model weights

        epoch_loss += loss.item()
        epoch_acc += accuracy

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

def evaluate(model, iterator, criterion, device):
    epoch_loss = 0
    epoch_acc = 0
    model.eval() # Set the model to evaluation mode

    with torch.no_grad(): # Disable gradient calculations
        for X_batch, y_batch in iterator:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            # Calculate accuracy
            _, predicted_labels = torch.max(predictions, 1)
            correct_predictions = (predicted_labels == y_batch).sum().item()
            accuracy = correct_predictions / y_batch.shape[0]

            epoch_loss += loss.item()
            epoch_acc += accuracy

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

print("Starting training...")

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    print(f'Epoch: {epoch+1:02}')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\tVal. Loss: {val_loss:.3f} | Val. Acc: {val_acc*100:.2f}%')

print("Training complete.")

# Evaluate on test set after training
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

Using device: cuda
DataLoaders created with batch size 64.
Starting training...
Epoch: 01
	Train Loss: 1.457 | Train Acc: 65.92%
	Val. Loss: 0.501 | Val. Acc: 83.41%
Epoch: 02
	Train Loss: 0.655 | Train Acc: 79.49%
	Val. Loss: 0.426 | Val. Acc: 84.49%
Epoch: 03
	Train Loss: 0.573 | Train Acc: 81.37%
	Val. Loss: 0.405 | Val. Acc: 84.78%
Epoch: 04
	Train Loss: 0.532 | Train Acc: 82.29%
	Val. Loss: 0.394 | Val. Acc: 85.10%
Epoch: 05
	Train Loss: 0.498 | Train Acc: 83.10%
	Val. Loss: 0.386 | Val. Acc: 84.71%
Epoch: 06
	Train Loss: 0.476 | Train Acc: 83.50%
	Val. Loss: 0.381 | Val. Acc: 85.18%
Epoch: 07
	Train Loss: 0.455 | Train Acc: 84.08%
	Val. Loss: 0.375 | Val. Acc: 85.13%
Epoch: 08
	Train Loss: 0.442 | Train Acc: 84.32%
	Val. Loss: 0.372 | Val. Acc: 85.12%
Epoch: 09
	Train Loss: 0.427 | Train Acc: 84.71%
	Val. Loss: 0.369 | Val. Acc: 85.39%
Epoch: 10
	Train Loss: 0.416 | Train Acc: 84.98%
	Val. Loss: 0.367 | Val. Acc: 85.35%
Epoch: 11
	Train Loss: 0.406 | Train Acc: 85.29%
	Val. Loss:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch

# Ensure the model is on the correct device
model.to(device)

# Function to get predictions and true labels
def get_predictions_and_labels(model, iterator, device):
    all_labels = []
    all_predictions = []
    model.eval() # Set model to evaluation mode
    with torch.no_grad(): # Disable gradient calculations
        for X_batch, y_batch in iterator:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predictions = model(X_batch)
            _, predicted_labels = torch.max(predictions, 1)

            all_labels.extend(y_batch.cpu().numpy())
            all_predictions.extend(predicted_labels.cpu().numpy())
    return np.array(all_labels), np.array(all_predictions)

# Get true and predicted labels for the test set
y_true, y_pred = get_predictions_and_labels(model, test_loader, device)

print("Evaluating model performance on the test set...")

# 3. Calculate and print overall accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall Test Accuracy: {accuracy:.4f}")

# 4. Generate and print classification report
# We need target names for the classification report, which are the inverse-transformed labels
# label_encoder was defined earlier, OUTPUT_DIM was also defined.
target_names = label_encoder.inverse_transform(range(OUTPUT_DIM))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

# 5. Compute and display the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Since there are many classes (OUTPUT_DIM), visualizing the full confusion matrix as a heatmap
# might be too large and unreadable. Instead, we can summarize it or show a partial view.
# For now, we'll just print the shape and a small part of it.
print(f"\nConfusion Matrix (shape {cm.shape}):")
print(cm[:5, :5]) # Displaying a small portion of the confusion matrix

# For visualization, if the number of classes is manageable, e.g., < 50, a heatmap can be useful.
# For 713 classes, it's not practical to visualize directly.
# If we were to visualize for fewer classes or a subset:
# plt.figure(figsize=(15, 12))
# sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
# plt.xlabel('Predicted Label')
# plt.ylabel('True Label')
# plt.title('Confusion Matrix')
# plt.show()

print("Test set evaluation complete. Classification report and a partial confusion matrix displayed.")

Evaluating model performance on the test set...

Overall Test Accuracy: 0.8551

Classification Report:
                                                          precision    recall  f1-score   support

                                            Rare Disease       0.78      0.55      0.64        33
                               abdominal aortic aneurysm       1.00      1.00      1.00        28
                                        abdominal hernia       0.96      0.98      0.97        81
                                         abscess of nose       0.76      0.88      0.82        58
                                     abscess of the lung       1.00      1.00      1.00         4
                                  abscess of the pharynx       0.83      0.91      0.87        68
                                    acanthosis nigricans       0.67      0.67      0.67         6
                                               acariasis       0.88      1.00      0.93         7
              

In [ ]:
train_losses_history = []
val_losses_history = []
train_accuracies_history = []
val_accuracies_history = []

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    train_losses_history.append(train_loss)
    val_losses_history.append(val_loss)
    train_accuracies_history.append(train_acc)
    val_accuracies_history.append(val_acc)

    print(f'Epoch: {epoch+1:02}')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\tVal. Loss: {val_loss:.3f} | Val. Acc: {val_acc*100:.2f}%')

Epoch: 01
	Train Loss: 0.322 | Train Acc: 87.41%
	Val. Loss: 0.366 | Val. Acc: 85.16%
Epoch: 02
	Train Loss: 0.321 | Train Acc: 87.47%
	Val. Loss: 0.363 | Val. Acc: 85.12%
Epoch: 03
	Train Loss: 0.318 | Train Acc: 87.54%
	Val. Loss: 0.363 | Val. Acc: 85.21%
Epoch: 04
	Train Loss: 0.315 | Train Acc: 87.65%
	Val. Loss: 0.370 | Val. Acc: 85.20%
Epoch: 05
	Train Loss: 0.314 | Train Acc: 87.70%
	Val. Loss: 0.365 | Val. Acc: 85.14%
Epoch: 06
	Train Loss: 0.314 | Train Acc: 87.70%
	Val. Loss: 0.367 | Val. Acc: 85.20%
Epoch: 07
	Train Loss: 0.313 | Train Acc: 87.77%
	Val. Loss: 0.365 | Val. Acc: 85.22%
Epoch: 08
	Train Loss: 0.311 | Train Acc: 87.80%
	Val. Loss: 0.370 | Val. Acc: 85.07%
Epoch: 09
	Train Loss: 0.309 | Train Acc: 87.83%
	Val. Loss: 0.369 | Val. Acc: 85.16%
Epoch: 10
	Train Loss: 0.308 | Train Acc: 87.85%
	Val. Loss: 0.371 | Val. Acc: 85.06%
Epoch: 11
	Train Loss: 0.306 | Train Acc: 87.94%
	Val. Loss: 0.368 | Val. Acc: 85.03%
Epoch: 12
	Train Loss: 0.305 | Train Acc: 87.99%
	Val.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

trainable_params = count_parameters(model)
print(f"Total number of trainable parameters: {trainable_params:,}")

# Calculate model size in megabytes
# Assuming each parameter is a 32-bit float (4 bytes)
bytes_per_parameter = 4 # for float32
model_size_bytes = trainable_params * bytes_per_parameter
model_size_mb = model_size_bytes / (1024 * 1024)

print(f"Estimated model size: {model_size_mb:.4f} MB")

Total number of trainable parameters: 619,465
Estimated model size: 2.3631 MB


In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Define new hyperparameters
NEW_LEARNING_RATE = 0.0001 # Slightly reduced learning rate
NEW_BATCH_SIZE = 128     # Increased batch size
NEW_EMBEDDING_DIM = 256  # Increased embedding dimension
NEW_HIDDEN_DIM = 512     # Increased hidden dimension
NEW_DROPOUT_PROB = 0.5   # Increased dropout probability
NEW_N_EPOCHS = 35        # Increased number of epochs

print(f"New Hyperparameters:")
print(f"  Learning Rate: {NEW_LEARNING_RATE}")
print(f"  Batch Size: {NEW_BATCH_SIZE}")
print(f"  Embedding Dim: {NEW_EMBEDDING_DIM}")
print(f"  Hidden Dim: {NEW_HIDDEN_DIM}")
print(f"  Dropout Prob: {NEW_DROPOUT_PROB}")
print(f"  Epochs: {NEW_N_EPOCHS}")

# 2. Instantiate a new SymptomClassifier model
# Reuse VOCAB_SIZE, OUTPUT_DIM, MAX_SEQ_LEN from previous steps
new_model = SymptomClassifier(
    VOCAB_SIZE,
    NEW_EMBEDDING_DIM,
    NEW_HIDDEN_DIM,
    OUTPUT_DIM,
    MAX_SEQ_LEN,
    NEW_DROPOUT_PROB
)

# Move new model to device
new_model.to(device)
print(f'New model moved to device: {device}')
print(f'\nThe new model has {count_parameters(new_model):,} trainable parameters.')

# 3. Re-initialize the Adam optimizer with the new model's parameters
new_optimizer = optim.Adam(new_model.parameters(), lr=NEW_LEARNING_RATE)

# Reuse criterion from previous step

# Create new DataLoaders with the new batch size
train_loader_tuned = DataLoader(train_data, shuffle=True, batch_size=NEW_BATCH_SIZE)
val_loader_tuned = DataLoader(val_data, shuffle=False, batch_size=NEW_BATCH_SIZE)
test_loader_tuned = DataLoader(test_data, shuffle=False, batch_size=NEW_BATCH_SIZE)

print(f"New DataLoaders created with batch size {NEW_BATCH_SIZE}.")

# 4. Re-implement the training loop and store metrics
train_losses_history_tuned = []
val_losses_history_tuned = []
train_accuracies_history_tuned = []
val_accuracies_history_tuned = []

print("Starting training with new hyperparameters...")

for epoch in range(NEW_N_EPOCHS):
    train_loss_tuned, train_acc_tuned = train(new_model, train_loader_tuned, new_optimizer, criterion, device)
    val_loss_tuned, val_acc_tuned = evaluate(new_model, val_loader_tuned, criterion, device)

    train_losses_history_tuned.append(train_loss_tuned)
    val_losses_history_tuned.append(val_loss_tuned)
    train_accuracies_history_tuned.append(train_acc_tuned)
    val_accuracies_history_tuned.append(val_acc_tuned)

    print(f'Epoch: {epoch+1:02}')
    print(f'\tTrain Loss: {train_loss_tuned:.3f} | Train Acc: {train_acc_tuned*100:.2f}%')
    print(f'\tVal. Loss: {val_loss_tuned:.3f} | Val. Acc: {val_acc_tuned*100:.2f}%')

print("Training with tuned hyperparameters complete.")

# 5. Evaluate on validation set after retraining
val_loss_final_tuned, val_acc_final_tuned = evaluate(new_model, val_loader_tuned, criterion, device)
print(f'\nFinal Tuned Validation Loss: {val_loss_final_tuned:.3f} | Final Tuned Validation Acc: {val_acc_final_tuned*100:.2f}%')

# 6. Evaluate on test set after retraining
y_true_tuned, y_pred_tuned = get_predictions_and_labels(new_model, test_loader_tuned, device)

print("\nEvaluating model performance on the test set with tuned hyperparameters...")

accuracy_tuned = accuracy_score(y_true_tuned, y_pred_tuned)
print(f"\nOverall Tuned Test Accuracy: {accuracy_tuned:.4f}")

print("\nClassification Report (Tuned Model):")
print(classification_report(y_true_tuned, y_pred_tuned, target_names=target_names, zero_division=0))

print("Hyperparameter tuning and re-evaluation complete.")

New Hyperparameters:
  Learning Rate: 0.0001
  Batch Size: 128
  Embedding Dim: 256
  Hidden Dim: 512
  Dropout Prob: 0.5
  Epochs: 35
New model moved to device: cuda

The new model has 2,024,649 trainable parameters.
New DataLoaders created with batch size 128.
Starting training with new hyperparameters...
Epoch: 01
	Train Loss: 3.148 | Train Acc: 47.24%
	Val. Loss: 1.417 | Val. Acc: 74.91%
Epoch: 02
	Train Loss: 1.096 | Train Acc: 76.99%
	Val. Loss: 0.708 | Val. Acc: 82.52%
Epoch: 03
	Train Loss: 0.688 | Train Acc: 82.33%
	Val. Loss: 0.522 | Val. Acc: 84.48%
Epoch: 04
	Train Loss: 0.554 | Train Acc: 83.77%
	Val. Loss: 0.452 | Val. Acc: 84.78%
Epoch: 05
	Train Loss: 0.488 | Train Acc: 84.68%
	Val. Loss: 0.420 | Val. Acc: 85.12%
Epoch: 06
	Train Loss: 0.452 | Train Acc: 85.18%
	Val. Loss: 0.401 | Val. Acc: 85.22%
Epoch: 07
	Train Loss: 0.430 | Train Acc: 85.54%
	Val. Loss: 0.389 | Val. Acc: 85.28%
Epoch: 08
	Train Loss: 0.411 | Train Acc: 85.90%
	Val. Loss: 0.382 | Val. Acc: 85.51%
Epo

In [ ]:
import torch

# Define the path to save the model
model_save_path = 'tuned_symptom_classifier_model.pth'

# Save the state dictionary of the new_model
torch.save(new_model.state_dict(), model_save_path)

print(f"Model state dictionary saved successfully to: {model_save_path}")

Model state dictionary saved successfully to: tuned_symptom_classifier_model.pth


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure NLTK data is downloaded for the preprocessing function
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Define the neural network model (copied from cell a005c9eb to ensure it's available)
class SymptomClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, max_seq_len, dropout_prob=0.5):
        super(SymptomClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(max_seq_len * embedding_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        flattened = self.flatten(embedded)
        x = self.fc1(flattened)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Model parameters (ensure these match the parameters used for the tuned model)
# These variables were defined in previous cells and are available in the kernel state
VOCAB_SIZE_LOADED = VOCAB_SIZE
EMBEDDING_DIM_LOADED = NEW_EMBEDDING_DIM # Use NEW_EMBEDDING_DIM from tuned model
HIDDEN_DIM_LOADED = NEW_HIDDEN_DIM      # Use NEW_HIDDEN_DIM from tuned model
OUTPUT_DIM_LOADED = OUTPUT_DIM
MAX_SEQ_LEN_LOADED = MAX_SEQ_LEN
DROPOUT_PROB_LOADED = NEW_DROPOUT_PROB # Use NEW_DROPOUT_PROB from tuned model

# Instantiate the model
loaded_model = SymptomClassifier(
    VOCAB_SIZE_LOADED,
    EMBEDDING_DIM_LOADED,
    HIDDEN_DIM_LOADED,
    OUTPUT_DIM_LOADED,
    MAX_SEQ_LEN_LOADED,
    DROPOUT_PROB_LOADED
)

# Define the path to the saved model
model_save_path = 'tuned_symptom_classifier_model.pth'

# Load the state dictionary
# Ensure 'device' is defined from previous cells (e.g., device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
loaded_model.load_state_dict(torch.load(model_save_path, map_location=torch.device('cpu')))

# Move the loaded model to the correct device (CPU or CUDA) before inference
loaded_model.to(device)

# Set the model to evaluation mode (important for inference)
loaded_model.eval()

print("Model loaded successfully!")

# Sample input for testing
sample_symptom_text = "anxiety, shortness of breath, depressive symptoms, chest tightness, palpitations"

# 1. Preprocess the sample input (using the preprocess_text function defined earlier in cell 904e656d)
# Ensure the preprocess_text function is available in the environment, or re-define it if necessary.
# For safety, I'll include it again here, though it should be in the global scope from previous execution.


def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^A-z ]', '', text) # Remove punctuation, keeping only letters and spaces
    text = re.sub(r'\d+', '', text)    # Remove numbers
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered_tokens)

cleaned_sample_text = preprocess_text(sample_symptom_text)
print(f"Original sample: '{sample_symptom_text}'")
print(f"Cleaned sample:  '{cleaned_sample_text}'")

# 2. Convert to numerical ID sequence and pad
# Ensure vocabulary and MAX_SEQ_LEN are available
# MAX_SEQ_LEN was determined in df83f4ce, vocabulary in d5240566

def text_to_sequence_inference(text, vocabulary, max_len):
    words = text.split()
    sequence = [vocabulary.get(word, vocabulary['<unk>']) for word in words]
    if len(sequence) < max_len:
        sequence = sequence + [vocabulary['<pad>']] * (max_len - len(sequence))
    else:
        sequence = sequence[:max_len]
    return sequence

# Using MAX_SEQ_LEN from the training process, which is MAX_SEQ_LEN (12)
processed_sequence = text_to_sequence_inference(cleaned_sample_text, vocabulary, MAX_SEQ_LEN)
print(f"Processed sequence: {processed_sequence}")

# 3. Convert to PyTorch tensor
input_tensor = torch.tensor(processed_sequence, dtype=torch.long).unsqueeze(0) # Add batch dimension
print(f"Input tensor shape: {input_tensor.shape}")

# 4. Move input to the same device as the model
input_tensor = input_tensor.to(device)

# 5. Get prediction from the loaded model
# loaded_model.eval() # Ensure model is in evaluation mode - already set above during loading
with torch.no_grad():
    output = loaded_model(input_tensor)
    probabilities = F.softmax(output, dim=1)
    predicted_label_id = torch.argmax(probabilities, dim=1).item()

# 6. Interpret the prediction
predicted_disease = label_encoder.inverse_transform([predicted_label_id])[0]

print(f"\nPredicted Disease ID: {predicted_label_id}")
print(f"Predicted Disease: {predicted_disease}")


Model loaded successfully!
Original sample: 'anxiety, shortness of breath, depressive symptoms, chest tightness, palpitations'
Cleaned sample:  'anxiety shortness breath depressive symptoms chest tightness palpitations'
Processed sequence: [1, 1, 1, 1, 1, 1, 1, 221, 0, 0, 0, 0]
Input tensor shape: torch.Size([1, 12])

Predicted Disease ID: 270
Predicted Disease: gastrointestinal hemorrhage


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Redefine the model architecture exactly as it was during training
class SymptomClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, max_seq_len, dropout_prob=0.5):
        super(SymptomClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(max_seq_len * embedding_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(p=dropout_prob)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        flattened = self.flatten(embedded)
        x = self.fc1(flattened)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Model parameters (ensure these match the parameters used for the tuned model)
# These variables were defined in previous cells and are available in the kernel state
VOCAB_SIZE_LOADED = VOCAB_SIZE
EMBEDDING_DIM_LOADED = NEW_EMBEDDING_DIM # Use NEW_EMBEDDING_DIM from tuned model
HIDDEN_DIM_LOADED = NEW_HIDDEN_DIM      # Use NEW_HIDDEN_DIM from tuned model
OUTPUT_DIM_LOADED = OUTPUT_DIM
MAX_SEQ_LEN_LOADED = MAX_SEQ_LEN
DROPOUT_PROB_LOADED = NEW_DROPOUT_PROB # Use NEW_DROPOUT_PROB from tuned model

# Instantiate the model
loaded_model = SymptomClassifier(
    VOCAB_SIZE_LOADED,
    EMBEDDING_DIM_LOADED,
    HIDDEN_DIM_LOADED,
    OUTPUT_DIM_LOADED,
    MAX_SEQ_LEN_LOADED,
    DROPOUT_PROB_LOADED
)

# Define the path to the saved model
model_save_path = 'tuned_symptom_classifier_model.pth'

# Load the state dictionary
loaded_model.load_state_dict(torch.load(model_save_path, map_location=torch.device('cpu')))

# Move the loaded model to the correct device (CPU or CUDA)
loaded_model.to(device)

# Set the model to evaluation mode (important for inference)
loaded_model.eval()

print("Model loaded successfully!")
print("Loaded Model Architecture:")
print(loaded_model)

Model loaded successfully!
Loaded Model Architecture:
SymptomClassifier(
  (embedding): Embedding(330, 256, padding_idx=0)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=3072, out_features=512, bias=True)
  (bn1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=512, out_features=713, bias=True)
)


In [ ]:
# Sample input for a more common disease (e.g., Allergy) using snake_case terms
sample_symptom_text_common = "allergic_reaction, nasal_congestion, cough, sore_throat, runny_nose, itchy_eyes"

print(f"Original sample: '{sample_symptom_text_common}'")

# 1. Preprocess the sample input using the existing function
cleaned_sample_text_common = preprocess_text(sample_symptom_text_common)
print(f"Cleaned sample:  '{cleaned_sample_text_common}'")

# 2. Convert to numerical ID sequence and pad
processed_sequence_common = text_to_sequence_inference(cleaned_sample_text_common, vocabulary, MAX_SEQ_LEN)
print(f"Processed sequence: {processed_sequence_common}")

# 3. Convert to PyTorch tensor
input_tensor_common = torch.tensor(processed_sequence_common, dtype=torch.long).unsqueeze(0) # Add batch dimension
print(f"Input tensor shape: {input_tensor_common.shape}")

# 4. Move input to the same device as the model
input_tensor_common = input_tensor_common.to(device)

# 5. Get prediction from the loaded model
loaded_model.eval() # Ensure model is in evaluation mode
with torch.no_grad():
    output_common = loaded_model(input_tensor_common)
    probabilities_common = F.softmax(output_common, dim=1)
    predicted_label_id_common = torch.argmax(probabilities_common, dim=1).item()

# 6. Interpret the prediction
predicted_disease_common = label_encoder.inverse_transform([predicted_label_id_common])[0]

print(f"\nPredicted Disease ID: {predicted_label_id_common}")
print(f"Predicted Disease: {predicted_disease_common}")

Original sample: 'allergic_reaction, nasal_congestion, cough, sore_throat, runny_nose, itchy_eyes'
Cleaned sample:  'allergic_reaction nasal_congestion cough sore_throat runny_nose itchy_eyes'
Processed sequence: [11, 201, 51, 277, 1, 1, 0, 0, 0, 0, 0, 0]
Input tensor shape: torch.Size([1, 12])

Predicted Disease ID: 122
Predicted Disease: chronic obstructive pulmonary disease (copd)


In [ ]:
disease_counts = df['diseases'].value_counts()

print("--- Disease Class Distribution Analysis ---")

# 1. Get the value counts of the `diseases` column and 2. Sort these counts in ascending order
sorted_disease_counts = disease_counts.sort_values(ascending=True)

# 3. Print the total number of unique diseases
total_unique_diseases = len(disease_counts)
print(f"Total number of unique diseases: {total_unique_diseases}")

# 4. Print the minimum, maximum, and average occurrences of diseases
min_occurrence = disease_counts.min()
max_occurrence = disease_counts.max()
avg_occurrence = disease_counts.mean()
print(f"Minimum disease occurrence: {min_occurrence}")
print(f"Maximum disease occurrence: {max_occurrence}")
print(f"Average disease occurrence: {avg_occurrence:.2f}")

# 5. Display the top 20 most frequent diseases
print("\nTop 20 Most Frequent Diseases:")
print(disease_counts.head(20).to_string())

# 6. Display the bottom 20 least frequent diseases (low-support classes)
print("\nBottom 20 Least Frequent Diseases (Low-Support Classes):")
print(sorted_disease_counts.head(20).to_string())

--- Disease Class Distribution Analysis ---
Total number of unique diseases: 713
Minimum disease occurrence: 6
Maximum disease occurrence: 1219
Average disease occurrence: 346.35

Top 20 Most Frequent Diseases:
diseases
cystitis                          1219
vulvodynia                        1218
nose disorder                     1218
complex regional pain syndrome    1217
spondylosis                       1216
vaginal cyst                      1215
conjunctivitis due to allergy     1215
esophagitis                       1215
peripheral nerve disorder         1215
hypoglycemia                      1215
gastrointestinal hemorrhage       1214
diverticulitis                    1214
acute bronchitis                  1213
pneumonia                         1212
spontaneous abortion              1212
fungal infection of the hair      1212
sprain or strain                  1212
infectious gastroenteritis        1212
gout                              1211
strep throat                      1210


In [ ]:
import numpy as np
from sklearn.utils import class_weight

# 1. Calculate class weights for nn.CrossEntropyLoss based on the inverse frequency
# Get the unique classes and their counts from the training set labels (y_train)
# Using sklearn's compute_class_weight to get balanced weights
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train.cpu().numpy()), # Ensure all classes are covered
    y=y_train.cpu().numpy()
)

# Map the weights to the original label IDs
# The compute_class_weight returns weights in the order of sorted unique classes
# We need to create a full weight tensor indexed by the original label IDs (0 to OUTPUT_DIM-1)
full_class_weights = np.zeros(OUTPUT_DIM)
unique_classes_in_train = np.unique(y_train.cpu().numpy())
for i, class_label in enumerate(unique_classes_in_train):
    full_class_weights[class_label] = class_weights_array[i]

# 2. Convert these class weights into a PyTorch tensor and move it to the appropriate device
class_weights_tensor = torch.tensor(full_class_weights, dtype=torch.float).to(device)

print("Calculated class weights:")
print(class_weights_tensor)
print(f"Shape of class weights tensor: {class_weights_tensor.shape}")

Calculated class weights:
tensor([ 2.0575,  2.5037,  0.8517,  1.1943, 18.8916,  1.0137, 11.5449,  9.4458,
         4.1562,  0.6973,  0.3813,  0.2878,  0.2858,  0.3820, 41.5616,  1.6493,
         0.3820,  0.5068,  0.2866,  0.7668,  0.3820,  0.3806,  4.7229,  0.6904,
         9.8956,  0.6791,  0.9849,  0.5208,  1.4041,  0.3827,  9.0351,  1.0187,
         1.0288,  7.1658,  5.7724,  1.5280, 17.3173,  1.0443,  6.9269, 29.6868,
         8.6587,  0.3820,  0.5709,  0.2890,  0.7668, 41.5616,  1.4233,  4.9478,
         0.3841,  1.3494,  0.2862,  4.7229,  1.0237, 34.6346,  0.3834,  1.0767,
         3.5222,  0.9075,  4.8327,  5.0685,  1.4431,  0.7901,  1.0549,  7.1658,
         0.6813,  0.7668,  4.1562, 34.6346,  0.7640,  0.5056,  1.6759,  0.7812,
         0.2870,  0.5068,  0.5234, 29.6868,  2.7708,  0.7640,  1.0237,  0.3820,
        51.9520,  4.1562,  1.3582, 10.9373,  0.3813,  1.5168, 51.9520, 14.8434,
         0.6904,  2.0991,  9.4458,  6.1120,  0.2866,  0.6747,  3.9209,  0.9991,
         1.111

**Reasoning**:
To apply cost-sensitive learning, I will re-instantiate the `nn.CrossEntropyLoss` criterion with the previously calculated `class_weights_tensor` to properly weight the loss contribution of each class during training.



In [ ]:
import torch.nn as nn

# 3. Re-instantiate the nn.CrossEntropyLoss criterion with the class weights
criterion_weighted = nn.CrossEntropyLoss(weight=class_weights_tensor)

print("Weighted CrossEntropyLoss criterion initialized.")

Weighted CrossEntropyLoss criterion initialized.


In [ ]:
import torch.optim as optim

# 1. Initialize a new model_weighted using the same architecture and hyperparameters as the new_model
# Reuse VOCAB_SIZE, OUTPUT_DIM, MAX_SEQ_LEN from previous steps
model_weighted = SymptomClassifier(
    VOCAB_SIZE,
    NEW_EMBEDDING_DIM,
    NEW_HIDDEN_DIM,
    OUTPUT_DIM,
    MAX_SEQ_LEN,
    NEW_DROPOUT_PROB
)

# 2. Move model_weighted to the appropriate device
model_weighted.to(device)
print(f'Weighted model moved to device: {device}')
print(f'The weighted model has {count_parameters(model_weighted):,} trainable parameters.')

# 3. Re-initialize an Adam optimizer (optimizer_weighted) for model_weighted
optimizer_weighted = optim.Adam(model_weighted.parameters(), lr=NEW_LEARNING_RATE)

print("Weighted model and optimizer initialized successfully.")

Weighted model moved to device: cuda
The weighted model has 2,024,649 trainable parameters.
Weighted model and optimizer initialized successfully.


**Reasoning**:
To implement cost-sensitive learning, I will modify the training loop to use the `criterion_weighted` loss function, train the `model_weighted` for the specified number of epochs, store the training and validation metrics, and then evaluate its performance on the test set, including a classification report to highlight improvements in low-support classes.



In [ ]:
train_losses_history_weighted = []
val_losses_history_weighted = []
train_accuracies_history_weighted = []
val_accuracies_history_weighted = []

print("Starting training with weighted loss...")

for epoch in range(NEW_N_EPOCHS):
    # 4. Modify the training loop to use criterion_weighted
    train_loss_weighted, train_acc_weighted = train(model_weighted, train_loader_tuned, optimizer_weighted, criterion_weighted, device)
    val_loss_weighted, val_acc_weighted = evaluate(model_weighted, val_loader_tuned, criterion_weighted, device)

    # 6. Store the training and validation metrics
    train_losses_history_weighted.append(train_loss_weighted)
    val_losses_history_weighted.append(val_loss_weighted)
    train_accuracies_history_weighted.append(train_acc_weighted)
    val_accuracies_history_weighted.append(val_acc_weighted)

    print(f'Epoch: {epoch+1:02}')
    print(f'\tTrain Loss: {train_loss_weighted:.3f} | Train Acc: {train_acc_weighted*100:.2f}%')
    print(f'\tVal. Loss: {val_loss_weighted:.3f} | Val. Acc: {val_acc_weighted*100:.2f}%')

print("Training with weighted loss complete.")

# 7. After training, evaluate model_weighted on the test_loader_tuned
y_true_weighted, y_pred_weighted = get_predictions_and_labels(model_weighted, test_loader_tuned, device)

print("\nEvaluating model performance on the test set with weighted loss...")

overall_weighted_test_accuracy = accuracy_score(y_true_weighted, y_pred_weighted)
print(f"\nOverall Weighted Test Accuracy: {overall_weighted_test_accuracy:.4f}")

# 8. Generate and print a classification report
print("\nClassification Report (Weighted Model):")
print(classification_report(y_true_weighted, y_pred_weighted, target_names=target_names, zero_division=0))

# 9. Compare the performance metrics (especially for low-support classes)
print("\n--- Performance Comparison (Weighted vs. Tuned Unweighted) ---")
print(f"Tuned Unweighted Model Test Accuracy: {accuracy_tuned:.4f}")
print(f"Weighted Model Test Accuracy: {overall_weighted_test_accuracy:.4f}")
print("Review the classification reports above for detailed class-wise comparison, especially for low-support classes.")


Starting training with weighted loss...
Epoch: 01
	Train Loss: 4.409 | Train Acc: 39.97%
	Val. Loss: 2.286 | Val. Acc: 72.82%
Epoch: 02
	Train Loss: 1.604 | Train Acc: 72.32%
	Val. Loss: 0.943 | Val. Acc: 81.03%
Epoch: 03
	Train Loss: 0.838 | Train Acc: 79.28%
	Val. Loss: 0.612 | Val. Acc: 83.57%
Epoch: 04
	Train Loss: 0.601 | Train Acc: 81.72%
	Val. Loss: 0.497 | Val. Acc: 84.41%
Epoch: 05
	Train Loss: 0.497 | Train Acc: 82.91%
	Val. Loss: 0.453 | Val. Acc: 84.86%
Epoch: 06
	Train Loss: 0.442 | Train Acc: 83.78%
	Val. Loss: 0.416 | Val. Acc: 85.18%
Epoch: 07
	Train Loss: 0.407 | Train Acc: 84.26%
	Val. Loss: 0.396 | Val. Acc: 85.34%
Epoch: 08
	Train Loss: 0.384 | Train Acc: 84.62%
	Val. Loss: 0.387 | Val. Acc: 85.39%
Epoch: 09
	Train Loss: 0.367 | Train Acc: 84.93%
	Val. Loss: 0.390 | Val. Acc: 85.45%
Epoch: 10
	Train Loss: 0.350 | Train Acc: 85.27%
	Val. Loss: 0.380 | Val. Acc: 85.54%
Epoch: 11
	Train Loss: 0.341 | Train Acc: 85.57%
	Val. Loss: 0.373 | Val. Acc: 85.59%
Epoch: 12
	Tra

In [ ]:
import collections

# Get all unique words from the cleaned_symptom column
all_cleaned_words = []
for text in df['cleaned_symptom']:
    all_cleaned_words.extend(text.split())

unique_cleaned_words = sorted(list(set(all_cleaned_words)))

# Identify words not present in the vocabulary
unknown_words = [word for word in unique_cleaned_words if word not in vocabulary]

print(f"Total unique words in cleaned symptoms: {len(unique_cleaned_words)}")
print(f"Number of words mapping to '<unk>' (not in vocabulary): {len(unknown_words)}")

if len(unknown_words) > 0:
    print("\nWords mapping to '<unk>':")
    # Display the first 50 unknown words for brevity
    for i, word in enumerate(unknown_words):
        if i < 50:
            print(f"- {word}")
        else:
            print(f"... (and {len(unknown_words) - 50} more)")
            break
else:
    print("\nNo words found that map to '<unk>'! All cleaned symptoms are in the vocabulary.")


Total unique words in cleaned symptoms: 328
Number of words mapping to '<unk>' (not in vocabulary): 0

No words found that map to '<unk>'! All cleaned symptoms are in the vocabulary.


In [ ]:
import pickle

# Define paths for saving
label_encoder_path = 'label_encoder.pkl'
vocabulary_path = 'vocabulary.pkl'
max_seq_len_path = 'max_seq_len.pkl'

# Save the LabelEncoder
with open(label_encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"LabelEncoder saved to: {label_encoder_path}")

# Save the vocabulary dictionary
with open(vocabulary_path, 'wb') as f:
    pickle.dump(vocabulary, f)
print(f"Vocabulary saved to: {vocabulary_path}")

# Save MAX_SEQ_LEN
with open(max_seq_len_path, 'wb') as f:
    pickle.dump(MAX_SEQ_LEN, f)
print(f"MAX_SEQ_LEN saved to: {max_seq_len_path}")

print("All essential components (LabelEncoder, vocabulary, MAX_SEQ_LEN) saved successfully.")


LabelEncoder saved to: label_encoder.pkl
Vocabulary saved to: vocabulary.pkl
MAX_SEQ_LEN saved to: max_seq_len.pkl
All essential components (LabelEncoder, vocabulary, MAX_SEQ_LEN) saved successfully.


### Loading essential components for deployment (Example)

In [ ]:
import pickle

# Define paths for loading
label_encoder_path = 'label_encoder.pkl'
vocabulary_path = 'vocabulary.pkl'
max_seq_len_path = 'max_seq_len.pkl'

# Load the LabelEncoder
with open(label_encoder_path, 'rb') as f:
    loaded_label_encoder = pickle.load(f)
print(f"LabelEncoder loaded from: {label_encoder_path}")

# Load the vocabulary dictionary
with open(vocabulary_path, 'rb') as f:
    loaded_vocabulary = pickle.load(f)
print(f"Vocabulary loaded from: {vocabulary_path}")

# Load MAX_SEQ_LEN
with open(max_seq_len_path, 'rb') as f:
    loaded_max_seq_len = pickle.load(f)
print(f"MAX_SEQ_LEN loaded from: {max_seq_len_path}")

print("All essential components loaded successfully.")

# Example usage with loaded components
print(f"\nFirst 5 classes from loaded LabelEncoder: {loaded_label_encoder.inverse_transform([0, 1, 2, 3, 4])}")
print(f"First 5 items from loaded Vocabulary: {list(loaded_vocabulary.items())[:5]}")
print(f"Loaded MAX_SEQ_LEN: {loaded_max_seq_len}")


LabelEncoder loaded from: label_encoder.pkl
Vocabulary loaded from: vocabulary.pkl
MAX_SEQ_LEN loaded from: max_seq_len.pkl
All essential components loaded successfully.

First 5 classes from loaded LabelEncoder: ['Rare Disease' 'abdominal aortic aneurysm' 'abdominal hernia'
 'abscess of nose' 'abscess of the lung']
First 5 items from loaded Vocabulary: [('<pad>', 0), ('<unk>', 1), ('abdominal_distention', 2), ('abnormal_appearing_skin', 3), ('abnormal_breathing_sounds', 4)]
Loaded MAX_SEQ_LEN: 12


In [ ]:
# Sample input for a different condition (e.g., Arthritis) using snake_case terms
sample_symptom_arthritis = "joint_pain, joint_swelling, joint_stiffness, redness, warmth , cold ,chest pain"

print(f"Original sample: '{sample_symptom_arthritis}'")

# Make prediction using the defined function
predicted_disease_arthritis, confidence_arthritis = predict_disease(
    sample_symptom_arthritis,
    loaded_model,
    loaded_label_encoder,
    loaded_vocabulary,
    loaded_max_seq_len,
    device
)

print(f"\nInput Symptoms: '{sample_symptom_arthritis}'")
print(f"Predicted Disease: {predicted_disease_arthritis}")
print(f"Confidence: {confidence_arthritis:.4f}")

Original sample: 'joint_pain, joint_swelling, joint_stiffness, redness, warmth , cold ,chest pain'


NameError: name 'predict_disease' is not defined

# Task
Create a dedicated `predict_disease` function that takes raw symptom text, the loaded model, LabelEncoder, vocabulary, and max sequence length as input, then demonstrate its usage with a new example symptom.